In [1]:
!pip install -q google-genai pytest pandas google-cloud-aiplatform

In [2]:
PROJECT_ID = "qwiklabs-gcp-01-ca95818a69b6"
LOCATION = "us-central1"

MODEL_NAME = "gemini-2.5-flash"

In [3]:
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

In [4]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say hello in one short sentence."
)

print(response.text)

Hello!


In [5]:
CATEGORIES = [
    "Employment",
    "General Information",
    "Emergency Services",
    "Tax Related"
]

def classify_user_question(question: str) -> str:
    prompt = f"""
                Classify the user question into exactly one category.

                Categories:
                - Employment
                - General Information
                - Emergency Services
                - Tax Related

                Return only the category name.

                User question:
                {question}
                """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    category = response.text.strip()

    if category not in CATEGORIES:
        return "General Information"

    return category

In [7]:
test_questions = [
    "How do I apply for unemployment benefits?",
    "When is city hall open?",
    "There is flooding on my street. Who do I call?",
    "Where can I get help filing my property taxes?"
]

for question in test_questions:
    print(question, "=>", classify_user_question(question))

How do I apply for unemployment benefits? => Employment
When is city hall open? => General Information
There is flooding on my street. Who do I call? => Emergency Services
Where can I get help filing my property taxes? => Tax Related


In [8]:
def generate_social_post(event_type: str, details: str) -> str:
    prompt = f"""
You are a government communications officer.

Write a clear, concise social media post for the public.

Event type: {event_type}
Details: {details}

Requirements:
- Keep it under 280 characters
- Use simple, clear language
- Include a call to action if relevant
- Maintain a calm, authoritative tone

Return only the post text.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [9]:
print(generate_social_post(
    "Weather Emergency",
    "Heavy snowfall expected tonight. Roads may be unsafe. City crews are on standby."
))

print("\n---\n")

print(generate_social_post(
    "School Closing",
    "All public schools will be closed tomorrow due to severe weather conditions."
))

Heavy snowfall expected tonight. Roads may become unsafe. Avoid unnecessary travel. City crews are on standby. Stay safe!

---

Due to severe weather, all public schools will be closed tomorrow. Please stay safe and monitor official channels for further updates.


In [10]:
%%writefile test_challenge_three.py

from unittest.mock import Mock

CATEGORIES = [
    "Employment",
    "General Information",
    "Emergency Services",
    "Tax Related"
]

def validate_category(category):
    if category not in CATEGORIES:
        return "General Information"
    return category


def test_validate_category_valid():
    assert validate_category("Employment") == "Employment"
    assert validate_category("Emergency Services") == "Emergency Services"


def test_validate_category_invalid():
    assert validate_category("Random Category") == "General Information"


def test_social_post_length():
    sample_post = "City offices are closed Monday for the holiday. Regular hours resume Tuesday."
    assert len(sample_post) <= 280


def test_social_post_has_content():
    sample_post = "Heavy snow expected tonight. Avoid unnecessary travel and follow local updates."
    assert len(sample_post.strip()) > 0

Writing test_challenge_three.py


In [11]:
!pytest -q

....                                                                     [100%]
4 passed in 0.01s


In [12]:
!pip install -q "google-cloud-aiplatform[evaluation]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.9 MB/s eta 0:00:00


In [14]:
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate
import pandas as pd

vertexai.init(project=PROJECT_ID, location=LOCATION)

In [15]:
def generate_social_post_prompt_v1(event_type: str, details: str) -> str:
    return f"""
Write a government social media post.

Event type: {event_type}
Details: {details}

Keep it short and clear.
"""


def generate_social_post_prompt_v2(event_type: str, details: str) -> str:
    return f"""
You are a government communications officer.

Write a clear, calm, public-facing social media post.

Event type: {event_type}
Details: {details}

Requirements:
- Under 280 characters
- Plain language
- Calm and authoritative tone
- Include action steps when useful
- Avoid panic or speculation

Return only the post.
"""

In [16]:
eval_rows = [
    {
        "event_type": "Weather Emergency",
        "details": "Freezing rain expected tonight. Roads may be hazardous by morning."
    },
    {
        "event_type": "Holiday Closure",
        "details": "City offices will be closed Monday for the statutory holiday."
    },
    {
        "event_type": "School Closing",
        "details": "All public schools are closed tomorrow due to severe snowfall."
    }
]

results = []

for row in eval_rows:
    prompt_1 = generate_social_post_prompt_v1(row["event_type"], row["details"])
    prompt_2 = generate_social_post_prompt_v2(row["event_type"], row["details"])

    response_1 = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_1
    ).text.strip()

    response_2 = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_2
    ).text.strip()

    results.append({
        "event_type": row["event_type"],
        "details": row["details"],
        "prompt_v1_response": response_1,
        "prompt_v2_response": response_2
    })

eval_df = pd.DataFrame(results)
eval_df

,event_type,details,prompt_v1_response,prompt_v2_response
0,Weather Emergency,Freezing rain expected tonight. Roads may be h...,**WEATHER ALERT:** Freezing rain expected toni...,Weather Alert: Freezing rain expected tonight....
1,Holiday Closure,City offices will be closed Monday for the sta...,City offices will be closed Monday for the sta...,Reminder: City offices will be closed Monday f...
2,School Closing,All public schools are closed tomorrow due to ...,🚨 **SCHOOL CLOSURE ANNOUNCEMENT** 🚨\n\nAll pub...,All public schools are closed tomorrow due to ...


In [17]:
social_post_quality_metric = PointwiseMetric(
    metric="social_post_quality",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "clarity": "The post is easy for the public to understand.",
            "tone": "The post is calm, authoritative, and appropriate for government communication.",
            "actionability": "The post gives useful action steps when relevant.",
            "brevity": "The post is concise and suitable for social media."
        },
        rating_rubric={
            "5": "Excellent public announcement.",
            "4": "Good announcement with minor issues.",
            "3": "Adequate but could be clearer or more actionable.",
            "2": "Weak announcement with important missing details.",
            "1": "Poor announcement."
        }
    )
)

INFO:vertexai.evaluation.metrics.metric_prompt_template:The `input_variables` parameter is empty. Only the `response` column is used for computing this model-based metric.


In [19]:
eval_task_v1 = EvalTask(
    dataset=eval_df.rename(columns={"prompt_v1_response": "response"}),
    metrics=[social_post_quality_metric],
    experiment="challenge-three-social-post-eval"
)

eval_result_v1 = eval_task_v1.evaluate()
eval_result_v1.summary_metrics

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 3/3 [00:24<00:00,  8.24s/it]
INFO:vertexai.evaluation._evaluation:All 3 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:24.722144030998606 seconds


{'row_count': 3,
 'social_post_quality/mean': np.float64(4.0),
 'social_post_quality/std': 1.7320508075688772}

In [21]:
eval_task_v2 = EvalTask(
    dataset=eval_df.rename(columns={"prompt_v2_response": "response"}),
    metrics=[social_post_quality_metric],
    experiment="challenge-three-social-post-eval"
)

eval_result_v2 = eval_task_v2.evaluate()
eval_result_v2.summary_metrics

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 3/3 [01:15<00:00, 25.10s/it]
INFO:vertexai.evaluation._evaluation:All 3 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:75.32157471999744 seconds


{'row_count': 3,
 'social_post_quality/mean': np.float64(5.0),
 'social_post_quality/std': 0.0}

## Evaluation Summary

I compared two prompt versions for generating government social media posts.

Prompt v1 was shorter and less specific.

Prompt v2 included clearer constraints:
- under 280 characters
- calm government tone
- plain language
- action steps when useful
- no panic or speculation

Based on the evaluation scores, Prompt v2 is the preferred prompt because it gives Gemini clearer quality criteria for public-facing emergency and civic announcements.